# Illustrating BLIP: Captioning, Q&A, and Matching

**BLIP** (Bootstrapping Language-Image Pre-training) is a vision-language model that sits between the two other Images notebooks:

- Unlike CLIP, which only produces embeddings to rank text, BLIP can generate text — it writes captions and answers questions about an image.
- Unlike a large hosted vision-chat API, BLIP is small enough to run locally on a laptop, with task-specific model heads rather than one big chat interface.

A single BLIP backbone supports three jobs, each with its own head:

| Task | HF class | What it does |
|---|---|---|
| **Captioning** | `BlipForConditionalGeneration` | image → descriptive sentence |
| **Visual Q&A** | `BlipForQuestionAnswering` | image + question → short answer |
| **Image-text matching** | `BlipForImageTextRetrieval` | image + text → match score |

We'll demonstrate all three with HuggingFace `transformers`, then look at **BLIP-2**, which bolts a frozen LLM onto the image encoder for richer output.

## Set up

The base BLIP models are a few hundred MB each and download from the HuggingFace Hub on first use.

In [ ]:
from PIL import Image
from skimage import data 
import matplotlib.pyplot as plt
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

We use the same labeled sample images as in the CLIP notebook.

In [ ]:
images = {
    "cat":       Image.fromarray(data.chelsea()),       # a tabby cat
    "coffee":    Image.fromarray(data.coffee()),        # a cup of coffee
    "astronaut": Image.fromarray(data.astronaut()),     # a person (astronaut)
    "rocket":    Image.fromarray(data.rocket()),        # a rocket launching
    "grogu":     Image.open('grogu_clarinet.jpg').convert('RGB'),     # grogu holding a clarinet
}

fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Using device: {device}")

## Image captioning

This is a key use of BLIP. 

We load the captioning model and processor:
* the processor preprocesses the image and (optionally) tokenizes a text prompt
* the model generates tokens that the processor decodes back into a string

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

CAP_MODEL = "Salesforce/blip-image-captioning-base"

cap_processor = BlipProcessor.from_pretrained(CAP_MODEL)
cap_model = BlipForConditionalGeneration.from_pretrained(CAP_MODEL).to(device)
cap_model.eval()

print("Loaded", CAP_MODEL)

### Unconditional captioning

Here we just give the model an image and let it describe what it sees.

In [ ]:
def caption(image, prompt=None, max_new_tokens=30):
    if prompt is None:
        inputs = cap_processor(image, return_tensors="pt").to(device)
    else:
        inputs = cap_processor(image, prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = cap_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return cap_processor.decode(out[0], skip_special_tokens=True)

captions = {name: caption(img) for name, img in images.items()}

fig, axes = plt.subplots(1, len(images), figsize=(3.2 * len(images), 3.4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img); ax.axis("off")
    ax.set_title(captions[name], fontsize=9, wrap=True)
plt.tight_layout(); plt.show()

for name, cap in captions.items():
    print(f"{name:>6}: {cap}")

### Conditional (prompted) captioning

Here instead we seed the caption with a prefix and BLIP continues it. This steers the model toward a focus or phrasing you want - useful for templated captions.

In [ ]:
img = images["cat"]
# img = images["grogu"]
for prefix in ["a photograph of", "a close-up of", "an illustration showing"]:
    print(f"{prefix!r:32} -> {caption(img, prompt=prefix)}")

## Visual question answering (VQA)

We can alternatively use BLIP to answer natural-language questions about an image. Answers are typically short (a word or phrase) — BLIP-VQA is trained on the VQA dataset, which favors terse answers.

In [ ]:
from transformers import BlipForQuestionAnswering

VQA_MODEL = "Salesforce/blip-vqa-base"
vqa_processor = BlipProcessor.from_pretrained(VQA_MODEL)
vqa_model = BlipForQuestionAnswering.from_pretrained(VQA_MODEL).to(device)
vqa_model.eval()

def ask(image, question, max_new_tokens=10):
    inputs = vqa_processor(image, question, return_tensors="pt").to(device)
    with torch.no_grad():
        out = vqa_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return vqa_processor.decode(out[0], skip_special_tokens=True)

print("Loaded", VQA_MODEL, "\n")

questions = ["What animal is this?", "What color is it?", "Is it indoors or outdoors?"]
for q in questions:
    print(f"Q: {q}\nA: {ask(images['cat'], q)}\n")

Let's display a couple of images with their Q&A side by side.

In [ ]:
qa_demo = [
    ("coffee", "What is in the cup?"),
    ("coffee", "What color is the cup?"),
    ("rocket", "What is launching?"),
    ("rocket", "Is there smoke?"),
    ("grogu",  "What is grogu holding?"),
    ("grogu",  "What color is the clarinet?"),
    ("grogu",  "Is the clarinet a weapon?"),
]

fig, axes = plt.subplots(1, 3, figsize=(8, 3.6))
for ax, name in zip(axes, ["coffee", "rocket", "grogu"]):
    ax.imshow(images[name]); ax.axis("off")
    qas = [f"Q: {q}\nA: {ask(images[name], q)}" for n, q in qa_demo if n == name]
    ax.set_title("\n".join(qas), fontsize=9, loc="left")
plt.tight_layout(); plt.show()

## Image-text matching (ITM)

BLIP's third head scores how well a caption matches an image. The ITM head is a binary match/no-match classifier; a softmax over its two logits gives the probability that the text describes the image. 

This is handy for verifying or ranking candidate captions (e.g. picking the best of several generated ones,
or filtering noisy web captions — which is part of how BLIP was trained).

In [ ]:
from transformers import BlipForImageTextRetrieval

ITM_MODEL = "Salesforce/blip-itm-base-coco"
itm_processor = BlipProcessor.from_pretrained(ITM_MODEL)
itm_model = BlipForImageTextRetrieval.from_pretrained(ITM_MODEL).to(device)
itm_model.eval()

def match_probability(image, text):
    """Probability (0-1) that `text` describes `image`, from the ITM head."""

    inputs = itm_processor(image, text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = itm_model(**inputs)            # use_itm_head=True by default
    
    # itm_score has shape (batch, 2): [no-match, match]
    probs = torch.softmax(out.itm_score, dim=1)
    
    return probs[0, 1].item()

print("Loaded", ITM_MODEL)

In [ ]:
candidates = [
    "a tabby cat resting",
    "a cup of black coffee on a table",
    "a rocket lifting off with smoke",
    "a snowy mountain peak at sunset",
    "a jedi holding an instrument",
    "a weapon"
]

img = images["cat"]
# img = images["grogu"]
scored = sorted(((match_probability(img, c), c) for c in candidates), reverse=True)

print("Ranking candidate captions for the CAT image:\n")
for score, text in scored:
    bar = "#" * int(score * 30)
    print(f"{score:6.1%} | {bar:<30} | {text}")

BLIP also carries a CLIP-style contrastive head (image/text cosine similarity), reachable with `itm_model(**inputs, use_itm_head=False)`. The ITM head above is more accurate for fine-grained matching because it lets image and
text tokens attend to each other, at the cost of running the model once per pair (no precomputed-embedding shortcut like CLIP).

## Advancing to BLIP-2: connecting a frozen LLM

BLIP-2 keeps a frozen image encoder and a frozen LLM, training only a small bridging module (the Q-Former) between them. 

This results in much richer, more fluent captioning and zero-shot VQA, including free-form prompted generation.

The catch is size: `blip2-opt-2.7b` is several GB and really wants a GPU. The cell below is guarded by a flag so that it doesn't run by accident — flip it to `True` when you're on suitable hardware and comfortable downloading a larger model.

In [ ]:
RUN_BLIP2 = False  # set True on a GPU with enough memory (~6 GB+ in fp16, ~15 GB to store downloaded model)

if RUN_BLIP2:
    from transformers import Blip2Processor, Blip2ForConditionalGeneration

    b2_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
    b2_model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b",
        torch_dtype=torch.float16,
    ).to(device)
    b2_model.eval()

    img = images["coffee"]

    # Fluent captioning (no prompt)
    inputs = b2_processor(img, return_tensors="pt").to(device, torch.float16)
    out = b2_model.generate(**inputs, max_new_tokens=30)
    print("Caption:", b2_processor.decode(out[0], skip_special_tokens=True).strip())

    # Prompted / VQA style
    prompt = "Question: What is in the cup and what is it sitting on? Answer:"
    inputs = b2_processor(img, prompt, return_tensors="pt").to(device, torch.float16)
    out = b2_model.generate(**inputs, max_new_tokens=40)
    print("Answer: ", b2_processor.decode(out[0], skip_special_tokens=True).strip())

else:
    print("BLIP-2 skipped. Set RUN_BLIP2 = True to run it on suitable hardware.")

In [ ]:
if RUN_BLIP2:

    img = images["grogu"]

    # Fluent captioning (no prompt)
    inputs = b2_processor(img, return_tensors="pt").to(device, torch.float16)
    out = b2_model.generate(**inputs, max_new_tokens=30)
    print("Caption:", b2_processor.decode(out[0], skip_special_tokens=True).strip())

    # Prompted / VQA style
    prompt = "Question: What is grogu holding? Answer:"
    inputs = b2_processor(img, prompt, return_tensors="pt").to(device, torch.float16)
    out = b2_model.generate(**inputs, max_new_tokens=40)
    print("Answer: ", b2_processor.decode(out[0], skip_special_tokens=True).strip())

    prompt = "Question: Is there a weapon in this image, and if so, what is it? Answer:"
    inputs = b2_processor(img, prompt, return_tensors="pt").to(device, torch.float16)
    out = b2_model.generate(**inputs, max_new_tokens=40)
    print("Answer: ", b2_processor.decode(out[0], skip_special_tokens=True).strip())

else:
    print("BLIP-2 skipped. Set RUN_BLIP2 = True to run it on suitable hardware.")

## BLIP vs CLIP vs a vision-chat API

Three points on a spectrum:

| | **CLIP** | **BLIP** | **Vision-chat API** |
|---|---|---|---|
| Core ability | embed & rank | caption, VQA, match | open-ended chat |
| Generative? | no | yes (short) | yes (long, instructable) |
| Runs locally? | yes, tiny | yes, small | usually hosted/served |
| Best for | search, zero-shot tags, dedup | auto-captions, simple VQA, caption filtering | complex reasoning, multi-image, instructions |
| Cost | cheapest | cheap | highest |

**Rule of thumb:**
* Need a vector to search or classify? CLIP. 
* Need a short generated description or answer from a small local model? BLIP. 
* Need reasoning, long answers, or following detailed instructions? The vision-chat API.

## Tips & limitations

- **Pick the right checkpoint per task.** Captioning, VQA, and ITM are different fine-tuned models — don't expect the captioning model to answer questions well, or vice versa. There are `-base` and `-large` sizes; large is slower but noticeably better.
- **VQA answers are terse by design.** The training data rewards one- or two-word answers. For explanations, use BLIP-2 with a prompt, or a vision-chat API.
- **Generation knobs apply.** `model.generate(...)` takes the usual `num_beams`, `do_sample`, `temperature`, `max_new_tokens` — tune them for more diverse or longer captions.
- **Batch for throughput.** The processor accepts a list of images; batched `generate` + `batch_decode` is much faster than a Python loop on GPU.
- **Known weaknesses.** Counting, reading text in images, fine spatial relations, and unusual/specialized domains. BLIP can also hallucinate plausible-but-wrong details — verify when it matters.
- **Caching.** Models download to `~/.cache/huggingface` on first use; later runs are offline-capable once cached.